# 2. Generating Ground Truth Data

In [1]:
%load_ext autoreload
%autoreload 2
import dotenv

dotenv.load_dotenv(override=True)

True

In [2]:
from src import FaqHttpLoader

loader = FaqHttpLoader()
documents = loader.load()

In [3]:
print(documents[0]['id'])
print(documents[0]['question'])

a30ab34d
How do I submit homework?


Generating questions with structured output

In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.
""".strip()

In [ ]:
import os
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.environ.get('OPENROUTER_API_KEY'),
    base_url='https://openrouter.ai/api/v1',
)

def llm_structured(
    instructions,
    user_prompt,
    output_type,
    model='nvidia/nemotron-3-super-120b-a12b:free'
    ):
    """
    openai/gpt-4.1-mini
    nvidia/nemotron-3-super-120b-a12b:free
    """
    messages = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=output_type,
        max_tokens=1024,
    )

    return response.choices[0].message.parsed

In [18]:
import json

result = llm_structured(
    data_gen_instructions,
    json.dumps(documents[0]),
    Questions
)

print(result.questions)

['What steps should I follow to submit my homework?', 'Where do I find the homework assignments for the 2025 cohort?', 'How do I get the link to the homework submission form?', 'After I submit my homework, when can I view the official answers?', 'Do I need to host my code somewhere specific for homework submission?']


Parallel processing

In [19]:
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results


def process(doc):
    out = llm_structured(
        data_gen_instructions,
        json.dumps(doc),
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            'question': q,
            'course': doc['course'],
            'document': doc['id']
        })

    return results

Generate questions for all documents:

In [20]:
with ThreadPoolExecutor(max_workers=6) as pool:
    ground_truth = map_progress(pool, documents, process)

  0%|          | 0/1208 [00:00<?, ?it/s]

KeyboardInterrupt: 